In [3]:
# Configuración inicial del entorno
import sys
import os
from pathlib import Path

print("🔄 Configurando entorno...")

# Obtener el directorio raíz del proyecto
notebook_dir = Path().resolve()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    # Buscar el directorio con pyproject.toml
    project_root = notebook_dir
    while project_root != project_root.parent:
        if (project_root / "pyproject.toml").exists():
            break
        project_root = project_root.parent

print(f"📁 Directorio raíz: {project_root}")

# Cambiar al directorio raíz
os.chdir(project_root)
print(f"✓ Directorio de trabajo: {os.getcwd()}")

# Agregar al path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Verificar pyproject.toml
if (project_root / "pyproject.toml").exists():
    print("✓ pyproject.toml encontrado")
else:
    raise FileNotFoundError(f"pyproject.toml no encontrado en {project_root}")

print("✅ Configuración completada\n")


🔄 Configurando entorno...
📁 Directorio raíz: C:\Users\PC RST\Documents\GitHub\ML_ClashRoyale
✓ Directorio de trabajo: C:\Users\PC RST\Documents\GitHub\ML_ClashRoyale
✓ pyproject.toml encontrado
✅ Configuración completada



# 📘 Fase 3: Preparación de los Datos (CRISP-DM)

Este notebook documenta la **tercera fase de CRISP-DM: Preparación de los Datos**.  
Se relaciona directamente con el pipeline de *data_preparation*, que incluye:

- Selección de columnas relevantes.  
- Combinación de datasets (de distintos días).  
- Validación del dataset combinado.  
- Creación de un resumen de preparación de datos.


## 1. Selección de columnas relevantes

En esta etapa se filtran únicamente las variables necesarias para el análisis y futuros modelos.  
Por ejemplo:

- Identificadores de partida y jugador.  
- Resultado de la partida (victoria/derrota).  
- Cartas seleccionadas.  
- Rareza de las cartas.  
- Presencia de win condition.  
- Trofeos y niveles de jugadores.  


## 2. Combinación de datasets

Los datasets correspondientes a distintos días se combinan en un único dataset.  
Esto permite realizar análisis temporales y comparar la evolución del meta durante la temporada.


## 3. Validación del dataset combinado

Se verifica que el dataset final cumpla con los requisitos:

- No contiene valores nulos o duplicados críticos.  
- Tiene consistencia en los tipos de datos.  
- Los valores están en rangos válidos.  


## 4. Resumen de preparación de datos

El resumen final sintetiza los pasos realizados en la fase de preparación de datos:  
- Qué columnas se seleccionaron.  
- Cuántos registros se mantuvieron.  
- Resultados de la validación.  
- Aspectos a considerar antes de avanzar al análisis exploratorio o modelado.


In [4]:
# Cargar salidas del pipeline de data_preparation desde Kedro
# Nota: Esta celda requiere que la celda de configuración inicial se haya ejecutado

try:
    # Verificar que project_root está definido
    if 'project_root' not in globals():
        raise NameError("project_root no está definido. Ejecuta primero la celda de configuración inicial.")
    
    print("🔄 Inicializando Kedro...")
    
    # Importar y configurar Kedro
    from kedro.framework.session import KedroSession
    from kedro.framework.startup import bootstrap_project
    
    # Bootstrap del proyecto
    print("  - Bootstrap del proyecto...")
    metadata = bootstrap_project(project_root)
    print(f"  ✓ Proyecto: {metadata.project_name}")
    
    # Crear sesión de Kedro
    print("  - Creando sesión de Kedro...")
    session = KedroSession.create(project_path=project_root)
    context = session.load_context()
    
    # Obtener el catálogo
    catalog = context.catalog
    
    print("\n✅ Contexto de Kedro cargado exitosamente")
    
    # Cargar datos del catálogo
    print("\n📊 Cargando datos del pipeline de data_preparation...")
    
    datasets_loaded = {}
    for dataset_name in ["selected_datasets", "combined_dataset", 
                         "dataset_validation", "data_preparation_summary"]:
        try:
            data = catalog.load(dataset_name)
            datasets_loaded[dataset_name] = data
            print(f"✓ {dataset_name} cargado")
        except Exception as e:
            print(f"⚠ {dataset_name} no disponible: {str(e)[:80]}")
            datasets_loaded[dataset_name] = None
    
    # Asignar a variables
    selected_datasets = datasets_loaded.get("selected_datasets")
    combined_dataset = datasets_loaded.get("combined_dataset")
    dataset_validation = datasets_loaded.get("dataset_validation")
    summary = datasets_loaded.get("data_preparation_summary")
    
    # Mostrar resultados
    if combined_dataset is not None:
        print("\n📊 Dataset combinado:")
        try:
            display(combined_dataset.head())
        except:
            print(f"Shape: {combined_dataset.shape}")
            print(combined_dataset.head())
    
    if dataset_validation is not None:
        print("\n✅ Resumen de validación:")
        try:
            display(dataset_validation)
        except:
            print(dataset_validation)
    
    if summary is not None:
        print("\n📋 Resumen de preparación de datos:")
        try:
            display(summary)
        except:
            print(summary)
    
    print("\n💡 Nota: Si los datos no están disponibles, ejecuta primero:")
    print("   kedro run --pipeline=data_preparation")
    
except NameError as e:
    print(f"\n❌ Error: {e}")
    print("\n💡 Ejecuta primero la celda de configuración inicial")
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\n💡 Posibles soluciones:")
    print("   1. Verifica que ejecutaste la celda de configuración inicial")
    print("   2. Verifica que pyproject.toml existe")
    print("   3. Si los datos no están disponibles, ejecuta: kedro run --pipeline=data_preparation")
    import traceback
    traceback.print_exc()


🔄 Inicializando Kedro...
  - Bootstrap del proyecto...
  ✓ Proyecto: ML ClashRoyale
  - Creando sesión de Kedro...


[11/28/25 15:56:56] WARNING  c:\Users\PC                                                            warnings.py:112
                             RST\Documents\GitHub\ML_ClashRoyale\venv\Lib\site-packages\kedro\frame                
                             work\project\__init__.py:350: UserWarning: The                                        
                             'proyecto_ml_clashroyale.pipelines.nodes' module does not expose a                    
                             'create_pipeline' function, so no pipelines defined therein will be                   
                             returned by 'find_pipelines'.                                                         
                               warnings.warn(                                                                      
                                                                                                                   

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving plugin.py:243
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/configuration/telemetry.html                         


✅ Contexto de Kedro cargado exitosamente

📊 Cargando datos del pipeline de data_preparation...


[11/28/25 15:56:57] INFO     Loading data from selected_datasets (PickleDataset)...            data_catalog.py:1048

...


✅ Resumen de validación:



{
    'total_records': 5644203,
    'total_columns': 34,
    'missing_values': {
        'battle_id': 0,
        'average.startingTrophies': 0,
        'winner.tag': 0,
        'winner.startingTrophies': 0,
        'winner.trophyChange': 0,
        'loser.tag': 0,
        'loser.startingTrophies': 0,
        'loser.trophyChange': 0,
        'winner.card1.id': 0,
        'winner.card2.id': 0,
        'winner.card3.id': 0,
        'winner.card4.id': 0,
        'winner.card5.id': 0,
        'winner.card6.id': 0,
        'winner.card7.id': 0,
        'winner.card8.id': 0,
        'winner.cards.list': 0,
        'winner.common.count': 0,
        'winner.rare.count': 0,
        'winner.epic.count': 0,
        'winner.legendary.count': 0,
        'loser.card1.id': 0,
        'loser.card2.id': 0,
        'loser.card3.id': 0,
        'loser.card4.id': 0,
        'loser.card5.id': 0,
        'loser.card6.id': 0,
        'loser.card7.id': 0,
        'loser.card8.id': 0,
        'loser.cards.list


📋 Resumen de preparación de datos:



{
    'preparation_overview': {
        'phase': 'Fase 3 - Preparación de Datos',
        'total_records_combined': 5644203,
        'total_columns_selected': 34,
        'memory_usage': '3311.543201446533 MB'
    },
    'data_quality': {
        'duplicate_records': 0,
        'missing_values_summary': 0,
        'data_integrity_checks': {
            'battle_ids_complete': True,
            'winner_tags_complete': True,
            'loser_tags_complete': True
        }
    },
    'selected_features': {
        'card_columns_total': 16,
        'winner_card_columns': 8,
        'loser_card_columns': 8,
        'count_columns': ['common.count', 'rare.count', 'epic.count', 'legendary.count']
    },
    'dataset_statistics': {
        'unique_battle_ids': 2626517,
        'unique_winner_tags': 2584748,
        'unique_loser_tags': 2681343
    },
    'next_steps': [
        'Dataset listo para análisis de patrones',
        'Preparado para modelado de machine learning',
        'Columnas


💡 Nota: Si los datos no están disponibles, ejecuta primero:
   kedro run --pipeline=data_preparation
